# Áp dụng EHO ở nhiều bài toán

Đo chất lượng, chi phí và mức vi phạm cận. Giả thuyết là cấu hình cho kết quả tốt trên Rastrigin có thể cho hiệu quả khác khi áp dụng vào mô hình bơm với nhu cầu và cận khác.

In [ ]:
import os
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eho import EHOConfig
from objectives import rastrigin
from experiment_helpers.runner import (
    assert_reproducible, compare_configurations, results_frame, seeds_for,
)
from experiment_helpers.statistics import summarize_experiment_results

MODE = os.getenv("EHO_EXPERIMENT_MODE", "QUICK").upper()
seeds = seeds_for(MODE)
base = EHOConfig(population_size=20, max_generations=50)
print(MODE, seeds)
print(base)
assert_reproducible(rastrigin, base, seeds[0])
print("Kiểm tra tái lập cùng seed đã đạt.")

In [ ]:
from dataclasses import asdict
from objectives.data_loader import load_pump_scenarios
from experiment_helpers.runner import maximum_overshoot, pump_grid_reference, run_multiple_seeds
from experiment_helpers.plotting import plot_convergence, plot_rastrigin_contour

def report(groups):
    frame = results_frame(groups)
    summary = summarize_experiment_results(frame)
    display(summary)
    plot_convergence(groups)
    plt.show()
    return summary

## 1. Rastrigin hai chiều

So sánh hai kích thước quần thể cùng số thế hệ. Contour dùng seed đầu tiên đã công bố, còn bảng dùng mọi seed.

In [ ]:
two_d = {f"P={size}": EHOConfig(dimension=2, population_size=size) for size in (5, 20)}
two_results = compare_configurations(rastrigin, two_d, seeds)
report(two_results)
plot_rastrigin_contour(two_results["P=5"][0])
plt.show()

## 2. Rastrigin bốn chiều

So sánh cấu hình gần ví dụ sách, quần thể lớn, số thế hệ lớn, alpha và beta khác. Dùng giá trị mean cuối để chọn cấu hình thử chuyển sang bài toán bơm.

In [ ]:
book_config = EHOConfig()
four_d = {
    "Gần ví dụ sách": book_config,
    "P=20": replace(book_config, population_size=20),
    "G=100": replace(book_config, max_generations=100),
    "Alpha 0.7": replace(book_config, alpha=0.7),
    "Beta 0.3": replace(book_config, beta=0.3),
}
four_results = compare_configurations(rastrigin, four_d, seeds)
four_summary = report(four_results)
selected = four_summary["mean"].idxmin()
print("Cấu hình được chọn theo mean 4D:", selected)

## 3. Rastrigin nhiều chiều

Dùng cùng ngân sách và xem fitness, thời gian, độ phân tán khi số chiều tăng.

In [ ]:
report(compare_configurations(rastrigin, {
    f"{d}D": EHOConfig(dimension=d, population_size=20) for d in (4, 8, 12)
}, seeds));

## 4–5. Dữ liệu máy bơm giả lập và chuyển cấu hình

So sánh cấu hình cơ sở với cấu hình đã chọn trên Rastrigin, đổi số chiều và cận theo mô hình bơm. Lưới 81 × 81 là mốc hữu hạn trong cận, phụ thuộc độ phân giải. Xem từng seed và mức thiếu/dư riêng. Thang điểm của Rastrigin và máy bơm có ý nghĩa khác nhau.

In [ ]:
scenarios = load_pump_scenarios()
display(pd.DataFrame(asdict(scenario) for scenario in scenarios.values()))
for name, scenario in scenarios.items():
    configs = {
        label: replace(config, dimension=2, lower_bounds=0, upper_bounds=scenario.upper_bounds)
        for label, config in (("Cơ sở", base), ("Chuyển từ 4D", four_d[selected]))
    }
    groups = compare_configurations(scenario, configs, seeds)
    _, grid_fitness = pump_grid_reference(scenario)
    print(name, "| Mốc lưới:", round(grid_fitness, 4))
    summary = report(groups)
    display(pd.DataFrame(
        {"group": label, "seed": result.random_seed, **scenario.breakdown(result.best_position)}
        for label, results in groups.items() for result in results
    ))
    print("Cấu hình có mean thấp hơn trong mẫu:", summary["mean"].idxmin())
assert_reproducible(scenario, configs["Cơ sở"], seeds[0])

## 6. Phân tích vượt biên

(12.4) tạo khoảng vượt cận trên tới 1 ở phép tách. Những tọa độ ngoài cận cũng có thể xuất hiện ở thế hệ sau qua các cập nhật khác. Giả thuyết là cận hẹp làm tăng tần suất vượt cận. Dùng cùng tập seed và ghi tổng lượt cá thể, tọa độ, số thế hệ bị ảnh hưởng và độ vượt xa nhất.

In [ ]:
bounds = {"Chuẩn": EHOConfig(dimension=2, population_size=20),
          "Hẹp": EHOConfig(dimension=2, population_size=20, lower_bounds=-0.2, upper_bounds=0.2)}
groups = compare_configurations(rastrigin, bounds, seeds, save_population=True)
frame = results_frame(groups)
frame["max_overshoot"] = [maximum_overshoot(result) for results in groups.values() for result in results]
display(frame[["group", "seed", "violating_individuals", "violating_coordinates",
               "violating_generations", "max_overshoot", "best_in_bounds"]])
totals = frame.groupby("group")["violating_coordinates"].mean()
totals.plot.bar(ylabel="Tọa độ vượt cận trung bình mỗi lần chạy", rot=0)
plt.show()
print(f"Nhiều lượt vượt cận hơn trong mẫu: {totals.idxmax()}.")

## Kết luận

Đọc độ phù hợp của EHO qua fitness, ngân sách và tính khả thi trong từng bài toán. Với bơm, tổng lưu lượng và chi phí phạt giải thích nghiệm tốt hơn một số fitness đơn lẻ. Kết quả chuyển cấu hình chỉ mô tả các kịch bản và seed đã thử.